In [ ]:
import pandas as pd
# pd.set_option('display.max_colwidth', None)


df = pd.read_json("../outputs/Text-Tune-Small-v13-corrected.jsonl", lines=True)
print(len(df))
df.head()

# Filter the corrections to identify DPO candidates

In [ ]:
import Levenshtein

asterisk_rows = []
over_corrected_rows = []
heavy_rewrite_rows = []

for index, data in df.iterrows():
    original = data.get('original', '')
    corrupted = data.get('corrupted', '')
    model_corrected = data.get('model_corrected', '')

    if model_corrected.count('*') > corrupted.count('*'):
        data['flag_reason'] = 'Added extra asterisks'
        asterisk_rows.append(data)
        continue

    
    # Strategy A: The "False Positive" Over-correction
    # If the input was already perfect, but the model changed it anyway.
    if corrupted == original and model_corrected != original:
        data['flag_reason'] = 'Changed a perfect sentence'
        over_corrected_rows.append(data)
        continue

    # Strategy B: The "Heavy Rewrite" Over-correction
    # If the model changed WAY more characters than necessary.
    # We calculate the distance between what it SHOULD have done vs what it DID.
    expected_distance = Levenshtein.distance(corrupted, original)
    actual_distance = Levenshtein.distance(corrupted, model_corrected)
    
    # If the model made significantly more edits than the ground truth required
    # (e.g., changing 15 characters when only 3 were needed)
    if actual_distance > (expected_distance + 10): 
        data['flag_reason'] = f'Heavy rewrite (Expected edits: {expected_distance}, Actual edits: {actual_distance})'
        heavy_rewrite_rows.append(data)
        continue

In [ ]:
print(f"Total cases with added asterisks: {len(asterisk_rows)}")
print(f"Total cases of over-correction (false positives): {len(over_corrected_rows)}")
print(f"Total cases of heavy rewrite over-correction: {len(heavy_rewrite_rows)}")

In [ ]:
all_rows = asterisk_rows + over_corrected_rows + heavy_rewrite_rows
flagged_df = pd.DataFrame(all_rows)

## Construct DPO Pairs

In [ ]:
from src.prompts import get_inference_prompt_v6

flagged_df["prompt"] = flagged_df.apply(lambda row: get_inference_prompt_v6(row["corrupted"]), axis=1)
flagged_df.rename(columns={"original": "chosen", "model_corrected": "rejected"}, inplace=True)

flagged_df.info()

In [ ]:
flagged_df.head()

In [ ]:
flagged_df.to_json("../data/processed/dpo_dataset_3b.jsonl", orient="records", lines=True, force_ascii=False)